# Implementing Neural Networks Lesson (7/20/26)

---

*Codecademy — Deep Learning with TensorFlow Path: Implementing Neural Networks. Concise grad-student notes.*

## Introduction

A **neural network** (inspired by the human brain) learns from data to make decisions and solve complex problems — like any ML method, it processes data and adjusts its model to best predict a desired outcome. Two common ML tasks it's used for:

- **Classification** — given data + true labels/categories, predict the label for new examples. E.g. predict whether a fire will occur on a future day given historical fire-hazard factors.
- **Regression** — given data + true continuous values, predict a value for new examples. E.g. forecast a future stock price from historical market data.

Parametric models like neural networks are described by **parameters**: configuration variables representing the model's knowledge. Parameters are tweaked using training data, and performance is evaluated on hold-out **test data** the model hasn't seen during training.

**Training loop, at a glance:**
1. **Forward pass** — input data (features + true labels) flows through the **model** to produce **predicted outputs**.
2. Predicted outputs and true labels both feed into a **loss/cost function**, which quantifies how wrong the predictions are.
3. An **optimizer** uses that loss to compute updates.
4. **Backward pass** — those updates flow back through the model, adjusting its parameters.

This forward → loss → backward cycle repeats over the training data until the model's predictions are good enough.

## Main Components of a Neural Network Pipeline

| Component | Role |
|---|---|
| **Input data** | training data provided to the network |
| **Optimizer** | algorithm that adjusts the network's parameters, based on the training data, to perform the task |
| **Loss / cost function** | tells the optimizer how well it's doing on the training data, and which direction to adjust parameters in |
| **Evaluation metrics** | tell us how well the current model performs on validation data — e.g. **mean absolute error (MAE)** for regression tells us how far predictions are, on average, from the true values |

## Predicting Medical Costs: Loading the Data

The **Medical Cost Personal Datasets** dataset — seven columns:

| Column | Description | Data type |
|---|---|---|
| `age` | age of primary beneficiary | numerical/integer |
| `sex` | insurance contractor gender | integer (female = 1, male = 0) |
| `bmi` | body mass index | numerical/real |
| `children` | number of children covered by health insurance | numerical/integer |
| `smoker` | smoking or not | integer (true = 1, false = 0) |
| `region` | beneficiary's residential area in the US | categorical (northeast, northwest, southeast, southwest) |
| `charges` | individual medical costs billed by health insurance | numerical/real |

Goal: predict `charges` from the rest of the columns. Since `charges` is continuous, this is a **regression** task. (Worth pausing on: `sex` and `bmi` carry real ethical implications when used to price insurance — whether to include them as predictive features is a judgment call, not just a technical one.)

**Loading with pandas:**
- `pd.read_csv(...)` reads the CSV into a DataFrame.
- `dataset.iloc[:, 0:6]` slices out the first 6 columns as **features** (independent variables); `dataset.iloc[:, -1]` slices out the last column (`charges`) as the **label** (dependent variable).
- `.shape` returns `(n_samples, n_features)`.
- `.describe()` prints summary statistics (mean, std, min/max, quartiles) for numeric columns.

In [1]:
import pandas as pd

#load the dataset
dataset = pd.read_csv('insurance.csv')
#choose first 7 columns as features
features = dataset.iloc[:,0:6]
#choose the final column for prediction
labels = dataset.iloc[:,-1]

#print the number of features in the dataset
print("Number of features: ", features.shape[1])
#print the number of samples in the dataset
print("Number of samples: ", features.shape[0])
#see useful summary statistics for numeric features
print(features.describe())

#your code here below
print(labels.shape)
print(labels.describe())

Number of features:  6
Number of samples:  1338
               age          sex          bmi     children       smoker
count  1338.000000  1338.000000  1338.000000  1338.000000  1338.000000
mean     39.207025     0.494768    30.663397     1.094918     0.204783
std      14.049960     0.500160     6.098187     1.205493     0.403694
min      18.000000     0.000000    15.960000     0.000000     0.000000
25%      27.000000     0.000000    26.296250     0.000000     0.000000
50%      39.000000     0.000000    30.400000     1.000000     0.000000
75%      51.000000     1.000000    34.693750     2.000000     0.000000
max      64.000000     1.000000    53.130000     5.000000     1.000000
(1338,)
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64


## Data Preprocessing: One-Hot Encoding and Standardization

**One-hot encoding** — neural networks can't work with string data directly, so categorical features (like `region`) need to become numeric. One-hot encoding creates a binary column per category — `region`'s 4 categories become 4 binary columns (`region_northeast`, `region_northwest`, ...). Done with `pd.get_dummies(features)`.

**Train/test split** — train the model on a training set, evaluate on a held-out test set it never saw during training. `train_test_split(features, labels, test_size=0.33, random_state=42)`: `test_size` controls the held-out fraction, `random_state` controls the shuffle (for reproducibility).

**Standardization vs. normalization** — numerical features often live on very different scales (`age` spans [18, 64], `children` spans [0, 5]). Left alone, the optimizer may update some weights faster than others just because of scale, not importance.
- **Standardization** rescales to zero mean, unit variance (`StandardScaler`).
- **Normalization** rescales to a fixed range, usually [0, 1] (`Normalizer`).
- No universally correct choice — try both, keep whichever performs better.

**`ColumnTransformer`** applies a transformer to specific columns and passes the rest through unchanged:
```python
ct = ColumnTransformer([('normalize', Normalizer(), ['age', 'bmi', 'children'])], remainder='passthrough')
```
It returns a NumPy array, which we convert back to a DataFrame with `pd.DataFrame(arr, columns=features_train.columns)` to keep working with labeled data.

**Fit on train, transform on test** — the scaler is `fit_transform`'d on the training data only, then just `transform`'d on the test data. Fitting on test data too would leak information from test into train, since the scaler's parameters would "know about" test-set values.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer
from sklearn.compose import ColumnTransformer

# one-hot encode the categorical 'region' column
features = pd.get_dummies(features)
print(features.columns.tolist())
# -> region becomes 4 binary columns: region_northeast/northwest/southeast/southwest

# split into train/test sets
features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels, test_size=0.33, random_state=42)

# normalize the numeric columns, pass the rest through unchanged
ct = ColumnTransformer([('normalize', Normalizer(), ['age', 'bmi', 'children'])], remainder='passthrough')
features_train_norm = ct.fit_transform(features_train)
features_test_norm = ct.transform(features_test)

# convert back to DataFrames for readability
features_train_norm = pd.DataFrame(features_train_norm, columns=features_train.columns)
features_test_norm = pd.DataFrame(features_test_norm, columns=features_test.columns)

print(features_train_norm.head())

['age', 'sex', 'bmi', 'children', 'smoker', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']
        age       sex       bmi children smoker region_northeast  \
0  0.863808  0.503821       0.0        1      0             True   
1  0.740865  0.670578  0.037993        0      1            False   
2  0.827684  0.560894  0.018393        1      1            False   
3  0.500102  0.865966       0.0        1      0            False   
4   0.83269  0.553739       0.0        0      1            False   

  region_northwest region_southeast region_southwest  
0            False            False            False  
1            False            False             True  
2             True            False            False  
3            False            False             True  
4            False            False             True  


### Exercise: Standardize Instead of Normalize

Same idea as above, but with `StandardScaler` (zero mean, unit variance) in place of `Normalizer` — build a new `ColumnTransformer`, fit on train, transform both train and test, and convert back to DataFrames.

In [3]:
from sklearn.preprocessing import StandardScaler

my_ct = ColumnTransformer([('scale', StandardScaler(), ['age', 'bmi', 'children'])], remainder = 'passthrough')
features_train_scale =  my_ct.fit_transform(features_train)
features_test_scale =  my_ct.transform(features_test)
features_train_scale =  pd.DataFrame(features_train_scale, columns = features_train.columns)
features_test_scale =  pd.DataFrame(features_test_scale, columns = features_test.columns)
print(features_train_scale.describe())
print(features_test_scale.describe())

               age         sex         bmi  children  smoker region_northeast  \
count   896.000000  896.000000  896.000000       896     896              896   
unique   47.000000  455.000000    6.000000         2       2                2   
top      -1.494934    0.293843   -0.912607         0       0            False   
freq     53.000000   12.000000  383.000000       459     708              666   

       region_northwest region_southeast region_southwest  
count               896              896              896  
unique                2                2                2  
top               False            False            False  
freq                670              667              685  
               age         sex         bmi  children  smoker region_northeast  \
count   442.000000  442.000000  442.000000       442     442              442   
unique   47.000000  312.000000    6.000000         2       2                2   
top      -1.424533   -0.484495   -0.912607         

## Neural Network Model: `tf.keras.Sequential`

With preprocessing done, we build the model. The most common TensorFlow model is **Keras Sequential** — it builds a model layer-by-layer, step-by-step, and can only have one input tensor and one output tensor.

- Import: `from tensorflow.keras.models import Sequential`
- To keep things readable, model construction is typically wrapped in its own function, e.g. `design_model()`.
- `Sequential(name="my first model")` initializes an empty model — `name` is optional.
- `my_model.layers` accesses the model's layers — right after initialization, this list is empty; layers get added next.

In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
from tensorflow.keras.models import Sequential

def design_model(features):
    my_model = Sequential(name="my first model")
    return my_model

my_model = design_model(features_train)
print(my_model.layers)
# -> [] (empty — no layers added yet)

[]


## Neural Network Model: Layers

**Layers** are the building blocks of a network, each containing one or more neurons. Every layer has learnable **parameters**: a **weight matrix** and a **bias matrix**, tuned during training.

A fully-connected (**Dense**) layer, where every neuron connects to every neuron in the next layer, is created with:
```python
from tensorflow.keras import layers
layer = layers.Dense(3)   # 3 neurons/outputs
```

**Weight/bias matrix shapes** depend on both the input layer's neuron count and the output layer's neuron count:
- Weight matrix: `(#input_neurons, #output_neurons)` — rows = neurons feeding in (e.g. `#features` for the first layer), columns = neurons in this layer.
- Bias matrix: `(#output_neurons,)` — one bias per neuron in this layer.

Right after `layers.Dense(3)` is created, `layer.weights` is an **empty list** — the number of output neurons (3) is known, but not the number of input neurons, since no input has been seen yet.

Once an input flows through the layer (e.g. `layer(input)`), TensorFlow infers the input size automatically and builds the weight/bias matrices — `layer.weights` becomes populated with the actual shapes.

In [5]:
import tensorflow as tf
from tensorflow.keras import layers

layer = layers.Dense(3)   # 3 neurons
print(layer.weights)
# -> [] (empty — no input seen yet, so input size is unknown)

# num_samples x num_features, matching our actual preprocessed data
num_samples, num_features = features_train.shape
input_data = tf.ones((num_samples, num_features))
output = layer(input_data)

print("weights shape:", layer.weights[0].shape)
print("bias shape:   ", layer.weights[1].shape)
# -> weights shape: (num_features, 3), bias shape: (3,)

[]
weights shape: (9, 3)
bias shape:    (3,)


### Exercise: Varying Samples, Features, and Neurons

Experiment with the shapes: change `#samples` (1338 → 5000), `#features` (11 → 21), and `#neurons` (3 → 10) in the toy `tf.ones` example, and observe which shape each change affects.

- **#samples** only changes the *output's* row count — it never appears in the weight/bias shapes at all.
- **#features** sets the weight matrix's row count (the input dimension).
- **#neurons** sets the weight matrix's column count *and* the bias vector's length (the output dimension).

In [6]:
layer = layers.Dense(3) #3 is the number we chose

print(layer.weights) #we get empty weight and bias arrays because tensorflow doesn't know what the shape is of the input to this layer

# 5000 samples, 21 features
input = tf.ones((5000, 21))
# a fully-connected layer with 10 neurons
layer = layers.Dense(10)
# calculate the outputs
output = layer(input)
# print the weights
print(layer.weights)
# -> weight matrix: (21, 10) -- rows track #features, cols track #neurons
# -> bias vector:   (10,)    -- tracks #neurons only
# -> #samples (5000) doesn't show up in either shape

[]
[<Variable path=dense_2/kernel, shape=(21, 10), dtype=float32, value=[[ 0.33608222  0.13826025  0.07219887 -0.12463787 -0.430256    0.35937697
  -0.12820727 -0.1377483   0.11175054  0.24615473]
 [ 0.22686416  0.42751247 -0.33758572 -0.10737765 -0.13563687  0.39709038
   0.25907898  0.13983834 -0.08323783 -0.33318275]
 [-0.36534745  0.4171831   0.22429055  0.42211062 -0.17370245  0.3678773
   0.32222265 -0.07428795 -0.10492396  0.2633695 ]
 [-0.26695216 -0.36780134 -0.08249184  0.27522528  0.11865348  0.33217454
  -0.4383036   0.32034427  0.05303761  0.06682271]
 [ 0.3582489   0.15307075 -0.24409583  0.302711    0.32184255  0.4353671
  -0.20906042 -0.27201593  0.15170974  0.39730096]
 [-0.13452965 -0.34421152 -0.26303136 -0.0682725  -0.2247565   0.1400032
   0.42887646  0.4130327   0.08555222  0.18707389]
 [ 0.15689915  0.03128129  0.12186426  0.26433742  0.39140773  0.26412827
  -0.16093081 -0.00672892 -0.34242505  0.4283774 ]
 [-0.36458394  0.28536612  0.31373346 -0.15088558  0.293

## Neural Network Model: Input Layer

The **input layer** isn't a transformative layer — it's a placeholder that tells the model the shape of incoming data. In Keras it's `tf.keras.layers.InputLayer`:

```python
from tensorflow.keras.layers import InputLayer
my_input = InputLayer(input_shape=(15,))   # 15 features
```

`input_shape`'s first dimension is `#features` — the number of samples/batch size doesn't need to be specified. To avoid hard-coding, derive it from the data: `num_features = my_data.shape[1]`.

Add the input layer to a model with `my_model.add(my_input)`, and inspect the model with `my_model.summary()`. For an input-only model, the summary reports **0 trainable parameters** — confirming the input layer is just a placeholder, not a layer with weights/biases.

In [7]:
from tensorflow.keras.layers import InputLayer

#get the number of features/dimensions in the data
num_features = features_train.shape[1]
#without hard-coding
my_input = InputLayer(input_shape=(num_features,))

my_model.add(my_input)
print(my_model.summary())
# -> Total params: 0 -- the input layer is just a placeholder, no trainable weights

/home/plewis/.local/lib/python3.10/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "my first model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


## Neural Network Model: Output Layer

The output layer's shape depends on the task. For **regression**, we want one numerical prediction per sample — so the output is a single neuron, regardless of how many samples we feed in.

Here, we're predicting one number per data point (`charges`), so the output layer has exactly **one neuron**:

```python
from tensorflow.keras.layers import Dense
my_model.add(Dense(1))
```

No `input_shape` needed — TensorFlow infers it automatically from the previous layer's output size.

In [8]:
from tensorflow.keras.layers import Dense

my_model.add(Dense(1))
print(my_model.summary())
# -> now has trainable params: (num_features -> 1) weights + 1 bias

Model: "my first model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 1)              │            10 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10 (40.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

None


## Neural Network Model: Hidden Layers

Input layer + output layer alone is just **linear regression** — to capture non-linear relationships between inputs and outputs, we need **hidden layers** in between.

```python
from tensorflow.keras.layers import Dense
my_model.add(Dense(64, activation='relu'))
```

- **64 neurons** (2⁶) is a common choice — powers of 2 make computation more efficient.
- **`activation='relu'`** — the activation function applied to this layer's output. Several options exist (softmax, sigmoid, ...), but **ReLU** is a strong, widely-used default for hidden layers.

Every added layer brings its own weight/bias vectors, growing the parameter count. With 9 input features → 64 hidden neurons → 1 output neuron:
- Hidden layer's weight matrix: `(9, 64)` (9 inputs feeding 64 neurons), bias: `(64,)`.
- Output layer's weight matrix: `(64, 1)` (64 inputs feeding 1 neuron), bias: `(1,)`.

In [9]:
# putting it together: input -> hidden -> output, in the correct order
def design_model(features):
    model = Sequential(name="my_first_model")
    num_features = features.shape[1]
    model.add(InputLayer(input_shape=(num_features,)))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))
    return model

my_model = design_model(features_train)
print(my_model.summary())
# -> hidden layer: (9, 64) weights + (64,) bias = 640 params
# -> output layer: (64, 1) weights + (1,) bias  = 65 params
# -> total: 705 trainable params

Model: "my_first_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 705 (2.75 KB)

 Trainable params: 705 (2.75 KB)

 Non-trainable params: 0 (0.00 B)

None


### Exercise: 128-Unit Hidden Layer

Same `design_model()` shape, but with a wider hidden layer — 128 units instead of 64.

In [10]:
def design_model(features):
  model = Sequential(name = "my_first_model")
  input = InputLayer(input_shape=(features.shape[1],))
  #add the input layer
  model.add(input)
  #your code here
  model.add(Dense(128, activation='relu'))

  #adding an output layer to our model
  model.add(Dense(1))
  return model

#invoke the function for our model design
model = design_model(features_train)

#print the model summary here
print(model.summary())
# -> hidden layer: (9, 128) weights + (128,) bias = 1280 params
# -> output layer: (128, 1) weights + (1,) bias  = 129 params
# -> total: 1409 trainable params

Model: "my_first_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,409 (5.50 KB)

 Trainable params: 1,409 (5.50 KB)

 Non-trainable params: 0 (0.00 B)

None


## Optimizers

Keras offers several optimizers (SGD, **Adam**, RMSprop, ...) — algorithms that adjust the model's weights toward better performance.

```python
from tensorflow.keras.optimizers import Adam
opt = Adam(learning_rate=0.01)
```

The **learning rate** is a **hyperparameter** (governs the learning *process*, unlike model parameters — weights/bias — which the model uses to make predictions). It controls step size in parameter space:
- **Too high** → large jumps, may overshoot/miss the solution.
- **Too low** → learning is slow, may not converge in time.
- `0.01` is a common starting value.

**Compiling the model:**
```python
my_model.compile(loss='mse', metrics=['mae'], optimizer=opt)
```
- **`loss`** — measures learning success; lower is better. For regression, **MSE** (mean squared error) is the usual choice.
- **`metrics`** — additional quantities tracked during training, not optimized directly. **MAE** (mean absolute error) is tracked here because, unlike MSE, it's in the same units as the target (dollars) — so it directly tells us how many dollars off the predictions are, on average.

In [11]:
from tensorflow.keras.optimizers import Adam

# design_model() here is the 128-unit version from the exercise above
my_model = design_model(features_train)
opt = Adam(learning_rate=0.01)
my_model.compile(loss='mse', metrics=['mae'], optimizer=opt)
print(my_model.loss, my_model.optimizer)

mse <keras.src.optimizers.adam.Adam object at 0x7f45d8392e60>


### Exercise: Compiling Inside `design_model()`

Consolidate the whole build to this point — input, hidden, output layers, optimizer, and `.compile()` — into a single `design_model()` call, so the returned model is ready to train.

In [12]:
def design_model(features):
  model = Sequential(name = "my_first_model")
  input = InputLayer(input_shape=(features.shape[1],))
   #add an input layer
  model.add(input)
  #add a hidden layer with 128 neurons
  model.add(Dense(128, activation='relu'))
  #add an output layer
  model.add(Dense(1))
  #your code here
  opt = Adam(learning_rate=0.01)
  model.compile(loss='mse',  metrics=['mae'], optimizer=opt)

  return model

#invoke the function for our model design
model = design_model(features_train)
print(model.summary())

Model: "my_first_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,409 (5.50 KB)

 Trainable params: 1,409 (5.50 KB)

 Non-trainable params: 0 (0.00 B)

None


## Training and Evaluating the Model

**Training:**
```python
my_model.fit(my_data, my_labels, epochs=50, batch_size=3, verbose=1)
```
- `my_data` / `my_labels` — the training features and true labels.
- `epochs` — number of full passes through the training data. Training is iterative, so multiple passes are needed; the right number is dataset-dependent and is itself a tunable **hyperparameter**.
- `batch_size` — number of data points processed before each parameter update. Also a tunable hyperparameter.
- `verbose=1` shows a progress bar during training.

**Evaluating on the held-out test set** (data the model never saw during training):
```python
val_mse, val_mae = my_model.evaluate(my_data, my_labels, verbose=0)
```
`.evaluate()` returns the loss (`mse`) and any tracked metrics (`mae`) on the given data.

Lesson's result: MAE ≈ $3884.21 — on average, predictions are off by ~$3800. Whether that's "good" often needs domain expertise: is $3800 an acceptable error for pricing insurance? The modeling process doesn't stop at a single number — it continues with judgment about what's good enough.

In [13]:
# verbose=0 here to keep the notebook output short (lesson uses verbose=1 for a progress bar)
model.fit(features_train, labels_train, epochs=50, batch_size=3, verbose=0)

val_mse, val_mae = model.evaluate(features_test, labels_test, verbose=0)
print("MSE:", round(val_mse, 2))
print("MAE: $", round(val_mae, 2))
# -> MAE is in dollars: on average, how far off our charge predictions are

MSE: 36510520.0
MAE: $ 4012.14


### Exercise: Full Pipeline with a Fixed Seed

Same fit/evaluate pattern, but with a few changes: `tensorflow.random.set_seed(35)` for reproducibility, learning rate `0.1` (vs. `0.01` earlier), `epochs=40`, `batch_size=1`, and `verbose=1` during training.

In [14]:
import tensorflow

tensorflow.random.set_seed(35) #for the reproducibility of results

def design_model(features):
  model = Sequential(name = "my_first_model")
  #without hard-coding
  input = InputLayer(input_shape=(features.shape[1],))
  #add the input layer
  model.add(input)
  #add a hidden layer with 128 neurons
  model.add(Dense(128, activation='relu'))
  #add an output layer to our model
  model.add(Dense(1))
  opt = Adam(learning_rate=0.1)
  model.compile(loss='mse',  metrics=['mae'], optimizer=opt)
  return model

#invoke the function for our model design
model = design_model(features_train)
print(model.summary())

#fit the model using 40 epochs and batch size 1
# (verbose=0 here to keep the notebook output short -- exercise uses verbose=1)
model.fit(features_train, labels_train, epochs = 40, batch_size = 1, verbose = 0)

#evaluate the model on the test data
val_mse, val_mae = model.evaluate(features_test, labels_test, verbose = 0)

print("MAE: ", val_mae)

/home/plewis/.local/lib/python3.10/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "my_first_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,409 (5.50 KB)

 Trainable params: 1,409 (5.50 KB)

 Non-trainable params: 0 (0.00 B)

None


MAE:  2719.806884765625


## TL;DR

- A neural network pipeline: **input data → optimizer → loss/cost function → evaluation metrics**, wired together in a forward pass → loss → backward pass training loop.
- **Preprocessing:** one-hot encode categorical columns (`pd.get_dummies`), split into train/test (`train_test_split`), and rescale numeric columns with `ColumnTransformer` + `StandardScaler`/`Normalizer` — fit on train only, transform both, to avoid information leakage.
- **`tf.keras.Sequential`** builds a model layer-by-layer. `InputLayer` is a placeholder (0 trainable params); `Dense` layers carry weight matrices `(#inputs, #neurons)` and bias vectors `(#neurons,)`, inferred automatically from the previous layer's shape.
- **Hidden layers with a nonlinear activation** (`relu`) are what make a network more expressive than plain linear regression; the **output layer** shape matches the task — one neuron per prediction for regression.
- **Compiling** wires up the optimizer (e.g. `Adam(learning_rate=...)`, a tunable hyperparameter), the loss to minimize (`mse` for regression), and metrics to track (`mae`, interpretable in the target's own units — here, dollars).
- **Training** (`model.fit(..., epochs, batch_size)`) and **evaluation** (`model.evaluate(...)` on held-out test data) close the loop. On the medical-cost dataset, MAE landed around **$2700–3800** depending on learning rate/epochs/batch size — whether that's "good enough" is a domain judgment call, not just a number.